Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    get_trino_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    get_trino_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()


In [ ]:
# DBTITLE 1,Cell 1

import trino
import pandas as pd


query = """
WITH prompts AS (
    SELECT ticket_id, prompt, created_at FROM corp_analytics.krisha_chat_ct_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.trisha_ct_chat_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.esha_ct_tbl
),
prompts1 AS (
    SELECT
        ticket_id,
        MAX_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS last_prompt,
        MIN_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS first_prompt
    FROM prompts
    WHERE SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1) IS NOT NULL
    GROUP BY 1
),

base_data AS (
SELECT
    a.canonical_ticket_id,
    a.phone_number,
    a.name,
    a.email,
    a.user_selected_language,
    a.csat,
    CAST(a.first_agent_assigned_time AS timestamp) AS first_agent_assigned_time,
    CAST(a.resolved_time AS timestamp) AS resolved_time,
    a.overall_last_agent_type_for_sub_ticket AS overall_last_agent_type,
    a.ticket_group,
    CASE
        WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_disposition IS NOT NULL THEN obcs_l1_agent_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_disposition IS NOT NULL THEN l1_agent_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_disposition IS NOT NULL THEN l2_agent_disposition
        ELSE xz.origination_reason
    END AS agent_filled_reason,
    CASE
        WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_sub_disposition IS NOT NULL THEN obcs_l1_agent_sub_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_sub_disposition IS NOT NULL THEN l1_agent_sub_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_sub_disposition IS NOT NULL THEN l2_agent_sub_disposition
        ELSE xz.origination_sub_reason
    END AS agent_filled_sub_reason,
    a.last_l1_agent_name AS l1_agent_name,
    a.last_l2_agent_name AS l2_agent_name,
    a.bot_description,
    xz.origination_reason AS agent_disposition,
    xz.origination_sub_reason AS agent_sub_disposition,
    p1.first_prompt AS language_sel,
    a.ticket_created_time,
    a.ticket_state,
    JSON_EXTRACT_SCALAR(xz.meta_data, '$.source_screen') AS required_screen,

    CAST(
        (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp),
         CAST(a.resolved_time AS timestamp)) / 60) AS INTEGER
    ) AS aht_minutes,

    CAST(
        (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp),
         CAST(a.resolved_time AS timestamp)) % 60) AS INTEGER
    ) AS aht_seconds,

    ROW_NUMBER() OVER (
        PARTITION BY a.canonical_ticket_id
        ORDER BY a.ticket_created_time DESC
    ) AS rn,

    a.latest_qrfc AS qrfc

FROM corp_analytics.cx_sub_ticket_data_model_tbl a
LEFT JOIN crm_tickets.tickets xz ON a.ticket_id = xz.id
LEFT JOIN prompts1 p1 ON a.ticket_id = p1.ticket_id

WHERE
    (
        (day_of_week(current_date) = 1 AND
            DATE(SUBSTRING(a.ticket_created_time, 1, 10))
            BETWEEN current_date - INTERVAL '2' DAY
            AND current_date - INTERVAL '1' DAY
        )
        OR
        (day_of_week(current_date) <> 1 AND
            DATE(SUBSTRING(a.ticket_created_time, 1, 10))
            = current_date - INTERVAL '1' DAY
        )
    )

    AND a.overall_last_agent_type_for_sub_ticket IN
        ('SUPPORT_AGENT_L1', 'SUPPORT_AGENT_L2')

    AND a.ticket_group IN (
        'BBPS',
        'PERSONAL_LOAN',
        'DIGITAL_GOLD',
        'UPI',
        'CREDIT_LINE_UPI',
        'AMC',
        'GENERAL_INSURANCE',
        'HOME_LOAN_SALES',
        'MOTOR_INSURANCE'
    )

    AND a.source_channel = 'CHAT'
)

SELECT *
FROM base_data
WHERE rn = 1
  AND agent_filled_sub_reason IS NOT NULL
  AND TRIM(CAST(agent_filled_sub_reason AS VARCHAR)) <> ''
  AND agent_filled_sub_reason NOT IN (
      'Blank Call',
      'Call disconnected by customer',
      'Call Ended',
      'Issue not clear',
      'Number Busy',
      'RNR',
      'Spam',
      'Blank',
      'Other Product related'
  )
"""

df = get_trino_df(query)
if not df.empty:
    display(df)
# DBTITLE 1,Cell 1

import trino
import pandas as pd


query = """
WITH prompts AS (
    SELECT ticket_id, prompt, created_at FROM corp_analytics.krisha_chat_ct_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.trisha_ct_chat_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.esha_ct_tbl
),
prompts1 AS (
    SELECT
        ticket_id,
        MAX_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS last_prompt,
        MIN_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS first_prompt
    FROM prompts
    WHERE SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1) IS NOT NULL
    GROUP BY 1
),

base_data AS (
SELECT
    a.canonical_ticket_id,
    a.phone_number,
    a.name,
    a.email,
    a.user_selected_language,
    a.csat,
    CAST(a.first_agent_assigned_time AS timestamp) AS first_agent_assigned_time,
    CAST(a.resolved_time AS timestamp) AS resolved_time,
    a.overall_last_agent_type_for_sub_ticket AS overall_last_agent_type,
    a.ticket_group,
    CASE
        WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_disposition IS NOT NULL THEN obcs_l1_agent_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_disposition IS NOT NULL THEN l1_agent_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_disposition IS NOT NULL THEN l2_agent_disposition
        ELSE xz.origination_reason
    END AS agent_filled_reason,
    CASE
        WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_sub_disposition IS NOT NULL THEN obcs_l1_agent_sub_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_sub_disposition IS NOT NULL THEN l1_agent_sub_disposition
        WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_sub_disposition IS NOT NULL THEN l2_agent_sub_disposition
        ELSE xz.origination_sub_reason
    END AS agent_filled_sub_reason,
    a.last_l1_agent_name AS l1_agent_name,
    a.last_l2_agent_name AS l2_agent_name,
    a.bot_description,
    xz.origination_reason AS agent_disposition,
    xz.origination_sub_reason AS agent_sub_disposition,
    p1.first_prompt AS language_sel,
    a.ticket_created_time,
    a.ticket_state,
    JSON_EXTRACT_SCALAR(xz.meta_data, '$.source_screen') AS required_screen,

    CAST(
        (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp),
         CAST(a.resolved_time AS timestamp)) / 60) AS INTEGER
    ) AS aht_minutes,

    CAST(
        (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp),
         CAST(a.resolved_time AS timestamp)) % 60) AS INTEGER
    ) AS aht_seconds,

    ROW_NUMBER() OVER (
        PARTITION BY a.canonical_ticket_id
        ORDER BY a.ticket_created_time DESC
    ) AS rn,

    a.latest_qrfc AS qrfc

FROM corp_analytics.cx_sub_ticket_data_model_tbl a
LEFT JOIN crm_tickets.tickets xz ON a.ticket_id = xz.id
LEFT JOIN prompts1 p1 ON a.ticket_id = p1.ticket_id

WHERE
    (
        (day_of_week(current_date) = 1 AND
            DATE(SUBSTRING(a.ticket_created_time, 1, 10))
            BETWEEN current_date - INTERVAL '2' DAY
            AND current_date - INTERVAL '1' DAY
        )
        OR
        (day_of_week(current_date) <> 1 AND
            DATE(SUBSTRING(a.ticket_created_time, 1, 10))
            = current_date - INTERVAL '1' DAY
        )
    )

    AND a.overall_last_agent_type_for_sub_ticket IN
        ('SUPPORT_AGENT_L1', 'SUPPORT_AGENT_L2')

    AND a.ticket_group IN (
        'BBPS',
        'PERSONAL_LOAN',
        'DIGITAL_GOLD',
        'UPI',
        'CREDIT_LINE_UPI',
        'AMC',
        'GENERAL_INSURANCE',
        'HOME_LOAN_SALES',
        'MOTOR_INSURANCE'
    )

    AND a.source_channel = 'CHAT'
)

SELECT *
FROM base_data
WHERE rn = 1
  AND agent_filled_sub_reason IS NOT NULL
  AND TRIM(CAST(agent_filled_sub_reason AS VARCHAR)) <> ''
  AND agent_filled_sub_reason NOT IN (
      'Blank Call',
      'Call disconnected by customer',
      'Call Ended',
      'Issue not clear',
      'Number Busy',
      'RNR',
      'Spam',
      'Blank',
      'Other Product related'
  )
"""

df = get_trino_df(query)
if not df.empty:
    display(df)


In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1sshtYrCvZS7lLufFwacEu8ZczfHmk4_U9QpUeFnPTb4/edit?gid=1102365521#gid=1102365521"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)